In [ ]:
import pandas as pd

df = pd.read_csv("../data/raw/Fraud_Data.csv")

In [ ]:
# Drop raw datetime columns
df = df.drop(["signup_time", "purchase_time"], axis=1)

# Drop non-numeric columns (important)
df = df.select_dtypes(include=["int64", "float64"])

In [ ]:
df.dtypes

In [ ]:
X = df.drop("class", axis=1)
y = df["class"]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

In [ ]:
y_train.value_counts()
y_train_res.value_counts()

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_res, y_train_res)

In [ ]:
y_pred = lr.predict(X_test)
y_prob = lr.predict_proba(X_test)[:, 1]

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, average_precision_score

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print("AUC-PR:", average_precision_score(y_test, y_prob))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

rf.fit(X_train_res, y_train_res)

In [ ]:
y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_rf))
print(confusion_matrix(y_test, y_pred_rf))
print("AUC-PR:", average_precision_score(y_test, y_prob_rf))

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5)
scores = cross_val_score(rf, X, y, cv=cv, scoring="f1")

print(scores.mean(), scores.std())

In [ ]:
import joblib
import os

# 1. Make sure the 'models' directory exists
os.makedirs("../models", exist_ok=True)

# 2. Save your trained Random Forest model (rf) to the folder
# This is the line that actually creates the file!
joblib.dump(rf, "../models/random_forest_model.pkl")

print("Success! 'random_forest_model.pkl' has been created in your models folder.")

In [ ]:
import joblib
import os

# 1. Make sure the 'models' directory exists
os.makedirs("../models", exist_ok=True)

# 2. Save your trained Logistic Regression model (lr)
joblib.dump(lr, "../models/logistic_regression_model.pkl")

print("Success! 'logistic_regression_model.pkl' has been created in your models folder.")

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train_res, y_train_res)